In [ ]:
import cv2
import os
import numpy as np
from mtcnn import MTCNN
from tensorflow.keras.preprocessing.image import img_to_array

# Initialize face detector
detector = MTCNN()

# Path to folder containing DFDC videos chunk
#unzip the file

video_folder = '/content/drive/MyDrive/DFDC'
output_folder = '/content/drive/MyDrive/output_cropped_faces'
os.makedirs(output_folder, exist_ok=True)

def process_video(video_path, output_dir):
    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    success, frame = cap.read()
    while success:
        # Detect faces in the frame
        faces = detector.detect_faces(frame)
        for i, face in enumerate(faces):
            x, y, w, h = face['box']
            # Crop face from frame
            face_crop = frame[y:y+h, x:x+w]
            # Resize to 299x299
            face_resized = cv2.resize(face_crop, (299, 299), interpolation=cv2.INTER_AREA)
            # Convert BGR to RGB
            face_rgb = cv2.cvtColor(face_resized, cv2.COLOR_BGR2RGB)
            # Normalize pixel values to range [0, 1]
            face_normalized = face_rgb.astype('float32') / 255.0
            # Optionally convert to array if used for model input
            face_array = img_to_array(face_normalized)
            # Save the processed face image
            output_path = os.path.join(output_dir, f"{os.path.basename(video_path).split('.')[0]}_frame{frame_count}_face{i}.png")
            cv2.imwrite(output_path, cv2.cvtColor((face_normalized * 255).astype(np.uint8), cv2.COLOR_RGB2BGR))
        frame_count += 1
        success, frame = cap.read()
    cap.release()

# Process all videos in the folder
for video_file in os.listdir(video_folder):
    if video_file.endswith(".mp4") or video_file.endswith(".mov"):
        video_path = os.path.join(video_folder, video_file)
        process_video(video_path, output_folder)

In [3]:
!pip install mtcnn